In [ ]:
# Rag Pipeline
# !pip install -U -q bitsandbytes transformers accelerate sentence-transformers pymupdf faiss-cpu arxiv hf_xet
#
# print("Dependencies installed.")

In [ ]:
# Imports
import os
import glob
import fitz
import faiss
import torch
import requests
import re
import arxiv
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# 1. Data Setup
def setup_data():
    save_dir = "rag_papers"
    os.makedirs(save_dir, exist_ok=True)

    # --- PART 1: Seminal Papers ---
    print(">>> Step 1: Downloading Seminal Papers...")
    seminal_urls = {
        "BERT_Pretraining.pdf": "https://arxiv.org/pdf/1810.04805.pdf",
        "LoRA_Low_Rank.pdf": "https://arxiv.org/pdf/2106.09685.pdf",
        "Attention_Is_All_You_Need.pdf": "https://arxiv.org/pdf/1706.03762.pdf"
    }
    for name, url in seminal_urls.items():
        path = f"{save_dir}/{name}"
        if not os.path.exists(path):
            print(f"   Downloading {name}...")
            try:
                response = requests.get(url, timeout=10) # Added timeout
                with open(path, 'wb') as f:
                    f.write(response.content)
            except Exception as e:
                print(f"   Failed to download {name}: {e}")

    # --- PART 2: Recent ArXiv Papers ---
    print(">>> Step 2: Filling folder with recent ArXiv papers...")

    # Construct a robust client
    client = arxiv.Client(
        page_size=30,
        delay_seconds=3.0, # Be polite to API
        num_retries=3
    )

    # Search for LLM papers
    search = arxiv.Search(
        query = "cat:cs.CL AND (LLM OR Transformer)",
        max_results = 25,
        sort_by = arxiv.SortCriterion.SubmittedDate
    )

    count = 0
    # Loop through results
    for result in client.results(search):
        # 1. Create a Windows-safe filename
        # Allow only alphanumeric, spaces, dashes, underscores
        safe_title = re.sub(r'[^\w\s-]', '', result.title).strip().lower()
        # Replace spaces/dashes with underscores
        safe_title = re.sub(r'[-\s]+', '_', safe_title)
        # Truncate to 50 chars to avoid Windows path limit issues
        filename = f"{safe_title[:50]}.pdf"
        filepath = os.path.join(save_dir, filename)

        if not os.path.exists(filepath):
            print(f"   Downloading: {filename}...")
            try:
                # 2. Use direct HTTP request (More reliable than library)
                response = requests.get(result.pdf_url, timeout=15)

                if response.status_code == 200:
                    with open(filepath, 'wb') as f:
                        f.write(response.content)
                    count += 1
                else:
                    print(f"   Error: HTTP {response.status_code}")

            except Exception as e:
                print(f"   Error downloading {filename}: {e}")
        else:
            print(f"   Skipping (Exists): {filename}")
            count += 1 # Count existing files too

    # Final Check
    total_files = len(glob.glob(f"{save_dir}/*.pdf"))
    print(f"\n✅ Data Ready. Total Papers in '{save_dir}': {total_files}")
    if total_files < 20:
        print("⚠️ Warning: Download count is low. Check your internet connection or ArXiv availability.")

setup_data()

In [ ]:
# 2. Ingestion and Retrival
class RAGPipeline:
    def __init__(self):
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = None
        self.chunks = []

    def ingest_folder(self, folder_path):
        pdf_files = glob.glob(f"{folder_path}/*.pdf")
        print(f"\nProcessing {len(pdf_files)} PDFs...")

        all_chunks = []
        for pdf_path in pdf_files:
            try:
                doc = fitz.open(pdf_path)
                filename = os.path.basename(pdf_path)
                for page_num, page in enumerate(doc):
                    text = page.get_text().replace('\n', ' ')

                    # Fixed Chunking Strategy
                    words = text.split()
                    chunk_size = 250
                    overlap = 30
                    for i in range(0, len(words), chunk_size - overlap):
                        chunk_words = words[i:i + chunk_size]
                        if len(chunk_words) < 50: continue

                        all_chunks.append({
                            "text": " ".join(chunk_words),
                            "metadata": f"[{filename}, Page {page_num+1}]"
                        })
            except Exception as e:
                print(f"Skipping {pdf_path}: {e}")

        self.chunks = all_chunks
        print(f"Created {len(all_chunks)} chunks. Indexing...")

        embeddings = self.embed_model.encode([c['text'] for c in all_chunks], convert_to_numpy=True)
        d = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(d)
        self.index.add(embeddings)
        print("Index Ready.")

    def retrieve(self, query, k=5):
        q_vec = self.embed_model.encode([query], convert_to_numpy=True)
        _, indices = self.index.search(q_vec, k)
        return [self.chunks[idx] for idx in indices[0] if idx != -1]


In [ ]:
# 3. Generator (Qwen2.5-1.5B)
class QwenGenerator:
    def __init__(self):
        model_id = "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"\n>>> Loading Local Model: {model_id}...")

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Running on: {self.device.upper()}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map=self.device,
            trust_remote_code=True
        )

    # --- MODE 1: RAG (WITH CONTEXT) ---
    def generate_rag(self, query, context_chunks, task_type='qa'):
        # Format Context
        context_text = ""
        for i, chunk in enumerate(context_chunks):
            context_text += f"Source [{i+1}]: {chunk['text']}\n\n"

        system_msg = (
            "You are a strict academic research assistant. "
            "Answer the question based ONLY on the provided Context. "
            "You MUST cite the Source number like [1] for every claim you make."
        )
        if task_type == "summary":
            user_msg = (
                f"Context:\n{context_text}\n\n"
                f"Task: Write a single 'Related Work' paragraph about '{query}' summarizing these papers. "
                f"Cite sources."
            )
        else:
            user_msg = (
                f"Context:\n{context_text}\n\n"
                f"Question: {query}"
            )

        return self._run_inference(system_msg, user_msg)

    # --- MODE 2: VANILLA (NO CONTEXT) ---
    def generate_vanilla(self, query):
        system_msg = "You are a helpful AI assistant. Answer the question to the best of your knowledge."
        user_msg = f"Question: {query}"
        return self._run_inference(system_msg, user_msg)

    # --- INTERNAL HELPER ---
    def _run_inference(self, system_msg, user_msg):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=1000,
            temperature=0.3,
            do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        return self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
# 4. Main Execution
def run_pipeline():
    # 1. Ingest
    rag = RAGPipeline()
    rag.ingest_folder("rag_papers")

    # 2. Load Model
    llm = QwenGenerator()

    # 3. Test Queries
    questions = [
        # FACTUAL RECALL (Vanilla might know this, but RAG has proof)
        "What are the two pre-training tasks used in BERT?",

        # SPECIFIC DETAIL (Vanilla usually fails or hallucinates specific numbers)
        "What is the rank 'r' typically set to in LoRA experiments?",

        # DEFINITION (Vanilla gives generic definition, RAG gives paper definition)
        "How is 'Scaled Dot-Product Attention' defined mathematically?",

        # SYNTHESIS (Requires reading the new papers you downloaded)
        "What are some recent methods for efficient Large Language Model training?",

        # NEGATIVE CONSTRAINT (Vanilla will hallucinate, RAG should say 'Not found' if not in papers)
        "What does the dataset say about 'Quantum Neural Networks'?"
    ]

    print("\n=== Qwen2.5-1.5B RAG EVALUATION ===")

    for i, q in enumerate(questions):
        print(f"\n🔎 QUESTION {i+1}: {q}")
        print("-" * 60)

        # --- RUN VANILLA (NO RAG) ---
        print("🤖 VANILLA QWEN (No Context):")
        vanilla_ans = llm.generate_vanilla(q)
        print(f"💡 Answer:\n{vanilla_ans}")
        print("-" * 30)

        # --- RUN RAG (WITH CONTEXT) ---
        print("📚 RAG SYSTEM (With PDF Context):")
        docs = rag.retrieve(q, k=4)
        rag_ans = llm.generate_rag(q, docs)
        print(f"💡 Answer:\n{rag_ans}")

        print("=" * 60)

    print("\n=== RELATED WORK GENERATION ===")
    topic = "Efficient LLM Training"
    docs = rag.retrieve("LoRA BERT Transformer parameter efficiency", k=6)
    summary = llm.generate_rag(topic, docs, task_type="summary")
    print(f"📝 Summary:\n{summary}")

if __name__ == "__main__":
    run_pipeline()